# Task 3 — CompactBlurCNN and Label-Smoothing E5 Experiments

This notebook trains only the two predeclared E5 children. It does **not** retrain E1–E4.

1. Gender starts from accepted E1 and changes only the architecture to CompactBlurCNN.
2. Usage starts from accepted E2 and adds only label smoothing `epsilon=0.05`.

Both models start from random weights and use the saved five-fold split. Select a Colab GPU
runtime, then use Run All. The main Task 3 notebook applies the frozen gates afterward.


## 1. Mount Drive and load the submitted branch

Drive supplies the dataset, accepted parents, registry, and persistent E5 output.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)


In [2]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip().splitlines()
    if dirty:
        print("Local repository changes found:")
        for change in dirty:
            print(f"  {change}")
        print("Trying a safe fast-forward update. Git will stop before overwriting a local file.")
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"Repository ready: {REPO_DIR}")
print(f"Branch: {BRANCH}")
print(f"Commit: {commit}")


Mounted at /content/drive
$ git clone --branch task-3-gender-usage-classification --single-branch https://github.com/TrnLin/MLA2.git /content/MLA2
Repository ready: /content/MLA2
Branch: task-3-gender-usage-classification
Commit: 54cbbfb0eb695ab979435f3d47d3bc386c0477c9


## 2. Copy the teacher data onto the runtime disk

Training reads images from Colab's local disk. The archive keeps the repository folder structure.


In [3]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_suffixes = {".jpg", ".jpeg"}

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [name for name in names if Path(name).is_absolute() or ".." in Path(name).parts]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive has no teacher images in the expected folder.")
    current_images = sum(path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*"))
    needs_extract = current_images != expected_images or not all(path.is_file() for path in required_files)
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*"))
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found {actual_images:,}; "
        f"missing files: {missing_files}"
    )
print(f"Teacher data ready: {actual_images:,} images")


Extracting 44,441 teacher images...
Teacher data ready: 44,441 images


## 3. Resolve the accepted parents and verify E5

Gender resolves its five E1 folds. Usage resolves its five accepted E2 folds. The checks verify
the compact architecture budget and the exact Usage smoothing value without an optimiser step.


In [4]:
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for output_dir in (DRIVE_TASK_DIR / "experiments", DRIVE_TASK_DIR / "logs", DRIVE_TASK_DIR / "results"):
    output_dir.mkdir(parents=True, exist_ok=True)

from fashion.train.task3_experiments import (
    check_task3_child_setup,
    latest_completed_baseline_parent_run_ids,
    latest_completed_usage_e2_parent_run_ids,
)

gender_parent_run_ids = latest_completed_baseline_parent_run_ids(
    "gender", output_root=DRIVE_TASK_DIR
)
usage_parent_run_ids = latest_completed_usage_e2_parent_run_ids(
    output_root=DRIVE_TASK_DIR
)
gender_e5_check = check_task3_child_setup(
    "gender_compact_blur_cnn",
    parent_run_ids=gender_parent_run_ids,
    root=REPO_DIR,
    device_name="cuda",
)
usage_e5_check = check_task3_child_setup(
    "usage_label_smoothing",
    parent_run_ids=usage_parent_run_ids,
    root=REPO_DIR,
    device_name="cuda",
)

if gender_e5_check["parameter_count"] > 100_000:
    raise RuntimeError("CompactBlurCNN exceeds the frozen parameter budget")
if gender_e5_check["architecture_macs"] > 35_000_000:
    raise RuntimeError("CompactBlurCNN exceeds the frozen MAC budget")
if usage_e5_check["label_smoothing"] != 0.05:
    raise RuntimeError("Usage E5 label smoothing contract changed")

print("GPU:", gender_e5_check["environment"]["gpu"])
print("Gender E1 parents:", gender_parent_run_ids)
print("Usage E2 parents: ", usage_parent_run_ids)
print("Gender E5 parameters:", gender_e5_check["parameter_count"])
print("Gender E5 MACs:", gender_e5_check["architecture_macs"])
print("Usage E5 smoothing:", usage_e5_check["label_smoothing"])
print("Optimizer steps during checks:", gender_e5_check["optimizer_steps"], usage_e5_check["optimizer_steps"])


GPU: NVIDIA L4
Gender E1 parents: ('t3_baseline_gender_smallcnn_f0_s2753_e46cd00adf0a_20260830T082833Zf8c1e0', 't3_baseline_gender_smallcnn_f1_s2753_e46cd00adf0a_20260830T083708Z143950', 't3_baseline_gender_smallcnn_f2_s2753_e46cd00adf0a_20260830T084548Z5acbf0', 't3_baseline_gender_smallcnn_f3_s2753_e46cd00adf0a_20260830T085427Z5d34f9', 't3_baseline_gender_smallcnn_f4_s2753_e46cd00adf0a_20260830T090303Z41a843')
Usage E2 parents:  ('t3_usage_e2_class_balanced_ce_usage_smallcnn_f0_s2753_5461e048c3b3_20260830T115815Z356f6d', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f1_s2753_5461e048c3b3_20260830T120645Z6d08cd', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f2_s2753_5461e048c3b3_20260830T121514Zd58aa0', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f3_s2753_5461e048c3b3_20260830T122347Z2cb06e', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f4_s2753_5461e048c3b3_20260830T123218Z94db47')
Gender E5 parameters: 67069
Gender E5 MACs: 29504320
Usage E5 smoothing: 0.05
Optimizer steps during

## 4. Train Gender E5: CompactBlurCNN

This foreground cell trains folds 0–4. Only the architecture changes from Gender E1.


In [5]:
from fashion.train.task3_experiments import run_task3_child_cv

gender_e5_result = run_task3_child_cv(
    "gender_compact_blur_cnn",
    parent_run_ids=gender_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
gender_e5_result


[task3] starting five-fold experiment=t3_gender_compact_blur_cnn for target=gender
[task3] preparing target=gender fold=0: train=26,220, validation=6,553
[task3] fitting fold-training RGB statistics for target=gender fold=0
[task3] RGB statistics ready for target=gender fold=0
[task3] registered t3_gender_e5_compact_blur_cnn_gender_compactblurcnn_f0_s2753_029107022917_20260831T060941Z1f0fc7; the first optimiser step may now run
[task3] target=gender fold=0 epoch=1/30 train_loss=0.7536 train_macro_f1=0.3021 validation_loss=0.6691 validation_macro_f1=0.3171
[task3] target=gender fold=0 epoch=2/30 train_loss=0.5234 train_macro_f1=0.4163 validation_loss=0.4948 validation_macro_f1=0.4667
[task3] target=gender fold=0 epoch=3/30 train_loss=0.4510 train_macro_f1=0.5475 validation_loss=0.4930 validation_macro_f1=0.5735
[task3] target=gender fold=0 epoch=4/30 train_loss=0.4160 train_macro_f1=0.6007 validation_loss=0.4511 validation_macro_f1=0.5981
[task3] target=gender fold=0 epoch=5/30 train_lo

{'target': 'gender',
 'fold_run_ids': ['t3_gender_e5_compact_blur_cnn_gender_compactblurcnn_f0_s2753_029107022917_20260831T060941Z1f0fc7',
  't3_gender_e5_compact_blur_cnn_gender_compactblurcnn_f1_s2753_029107022917_20260831T061823Z81907a',
  't3_gender_e5_compact_blur_cnn_gender_compactblurcnn_f2_s2753_029107022917_20260831T062704Zf23c53',
  't3_gender_e5_compact_blur_cnn_gender_compactblurcnn_f3_s2753_029107022917_20260831T063549Za32b4e',
  't3_gender_e5_compact_blur_cnn_gender_compactblurcnn_f4_s2753_029107022917_20260831T064434Z46c915'],
 'prediction_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_e5_compact_blur_cnn/gender/aggregate/oof_predictions.csv',
 'metrics_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_e5_compact_blur_cnn/gender/aggregate/metrics.json',
 'class_report_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_e5_compact_blur_cnn/gender/aggregate/per_class.csv',
 'confusion_path': '/content/drive/MyDrive/MLA2/task3/expe

## 5. Train Usage E5: class-balanced label smoothing

This foreground cell trains folds 0–4. It keeps E2's architecture and class weights and changes
only the target distribution with `epsilon=0.05`.


In [6]:
usage_e5_result = run_task3_child_cv(
    "usage_label_smoothing",
    parent_run_ids=usage_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
usage_e5_result


[task3] starting five-fold experiment=t3_usage_label_smoothing_smallcnn for target=usage
[task3] preparing target=usage fold=0: train=26,219, validation=6,553
[task3] fitting fold-training RGB statistics for target=usage fold=0
[task3] RGB statistics ready for target=usage fold=0
[task3] registered t3_usage_e5_label_smoothing_usage_smallcnn_f0_s2753_0ccaa3f16434_20260831T065319Z338806; the first optimiser step may now run
[task3] target=usage fold=0 epoch=1/30 train_loss=1.3415 train_macro_f1=0.1608 validation_loss=1.2268 validation_macro_f1=0.2124
[task3] target=usage fold=0 epoch=2/30 train_loss=1.2021 train_macro_f1=0.2521 validation_loss=1.1316 validation_macro_f1=0.2491
[task3] target=usage fold=0 epoch=3/30 train_loss=1.0848 train_macro_f1=0.3035 validation_loss=1.1306 validation_macro_f1=0.2761
[task3] target=usage fold=0 epoch=4/30 train_loss=1.0021 train_macro_f1=0.3175 validation_loss=1.1342 validation_macro_f1=0.2880
[task3] target=usage fold=0 epoch=5/30 train_loss=0.9579 t

{'target': 'usage',
 'fold_run_ids': ['t3_usage_e5_label_smoothing_usage_smallcnn_f0_s2753_0ccaa3f16434_20260831T065319Z338806',
  't3_usage_e5_label_smoothing_usage_smallcnn_f1_s2753_0ccaa3f16434_20260831T070200Zba9896',
  't3_usage_e5_label_smoothing_usage_smallcnn_f2_s2753_0ccaa3f16434_20260831T071044Zb8469e',
  't3_usage_e5_label_smoothing_usage_smallcnn_f3_s2753_0ccaa3f16434_20260831T071926Z24011b',
  't3_usage_e5_label_smoothing_usage_smallcnn_f4_s2753_0ccaa3f16434_20260831T072807Z03061f'],
 'prediction_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e5_label_smoothing/usage/aggregate/oof_predictions.csv',
 'metrics_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e5_label_smoothing/usage/aggregate/metrics.json',
 'class_report_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e5_label_smoothing/usage/aggregate/per_class.csv',
 'confusion_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e5_label_smoothing/usage/aggregate/con

## 6. Show the first parent–child comparison

This is a smoke check only. The main Task 3 notebook must apply every frozen gate before either
E5 child can replace its accepted parent.


In [7]:
import pandas as pd

parent_metric_paths = {
    "gender": DRIVE_TASK_DIR / "baseline/gender/aggregate/metrics.json",
    "usage": DRIVE_TASK_DIR / "experiments/t3_usage_e2_class_balanced_ce/usage/aggregate/metrics.json",
}
children = {"gender": gender_e5_result, "usage": usage_e5_result}
comparison = []
for target, child in children.items():
    parent = json.loads(parent_metric_paths[target].read_text(encoding="utf-8"))
    comparison.append({
        "target": target,
        "parent_macro_f1": parent["macro_f1"],
        "e5_macro_f1": child["metrics"]["macro_f1"],
        "macro_f1_change": child["metrics"]["macro_f1"] - parent["macro_f1"],
        "model_family": child["metrics"]["model_family"],
        "parameter_count": child["metrics"]["parameter_count"],
        "architecture_macs": child["metrics"]["architecture_macs"],
        "e5_metrics_path": child["metrics_path"],
    })
pd.DataFrame(comparison)


,target,parent_macro_f1,e5_macro_f1,macro_f1_change,model_family,parameter_count,architecture_macs,e5_metrics_path
0,gender,0.711753,0.706854,-0.004899,task3_compact_blur_cnn,67069,29504320.0,/content/drive/MyDrive/MLA2/task3/experiments/...
1,usage,0.408171,0.404864,-0.003307,task3_small_cnn,391209,NaN,/content/drive/MyDrive/MLA2/task3/experiments/...
